In [1]:
from FIT_python.plot_style import apply_style

apply_style()

In [2]:
# %%
import time
from pathlib import Path
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
from tqdm_joblib import tqdm_joblib
from sklearn.model_selection import ParameterSampler
from FIT_python.config import SPLITS_DIR, RESULTS_DATA_DIR
from sklearn.model_selection import ParameterSampler
from FIT_python.data_split_and_summary.data_import_utils import (
    load_and_prep_df_individual,
    get_feature_cols,
)
from FIT_python.pipeline_individual_id.pairwise_individual_id_pipeline import (
    run_all_pairwise_projections_parallel,
)
from FIT_python.pipeline_individual_id.generate_trails_and_trailpairs import (
    generate_pairwise_comparisons_from_df,
)

import itertools

import matplotlib.pyplot as plt
import seaborn as sns

# %% 1) Basis-Pfade & Hyperparameter-Räume (unverändert)
splits_root = Path("data/splits/eurasian_otter")
models_root = (
    RESULTS_DATA_DIR
    / "eurasian_otter_random_search_standard_metrics"
    / "best_maj_test_pct"
)
output_root = Path("results/data/pairwise_comp_eurasian_otter")
output_root.mkdir(parents=True, exist_ok=True)

## %% 2) Statt Random-Sampling nur die Standard-Methoden

std_feature_counts = [10, 12, 14, 16, 18, 20]
std_selection_methods = ["forward", "random_forest"]
std_reducers = ["lda"]  # ggf. auch ["lda","pca","umap"]
std_n_components = [2]  # bei LDA nicht relevant, sonst Liste von ints
std_outlier_methods = [None]
std_scaler_methods = [None]
std_use_sex_opts = [False, True]

# Kreuzprodukt aller Standard-Settings
sampled_grid = pd.DataFrame(
    itertools.product(
        std_feature_counts,
        std_selection_methods,
        std_reducers,
        std_n_components,
        std_outlier_methods,
        std_scaler_methods,
        std_use_sex_opts,
    ),
    columns=[
        "feature_count",
        "selection_method",
        "reducer",
        "n_components",
        "outlier_method",
        "scaler_method",
        "use_sex",
    ],
)

# Sortieren optional
sampled_grid = sampled_grid.sort_values("feature_count", ascending=False).reset_index(
    drop=True
)

# Überprüfen
print(f"Standard grid shape: {sampled_grid.shape}")
display(sampled_grid)

# Überprüfen
print(f"Standard grid shape: {sampled_grid.shape}")
display(sampled_grid)


# 6) Loop über Spezies
species_dir = Path(SPLITS_DIR) / "eurasian_otter"
species = "eurasian_otter"
print(f"\n🔄 Processing species: {species}")

# 6.1) Train/Test laden & Fold entfernen
df = load_and_prep_df_individual(
    species, Path(SPLITS_DIR)
)  # falls hier bugs valid vales f und m checken


# 3.2) Feature-Spalten bestimmen
feature_cols = get_feature_cols(df)
print(f"  → {len(feature_cols)} numerische Features")

# direkt vor generate_pairwise_comparisons_from_df
print("Unique individual_id:", df["individual_id"].nunique())
print("Unique trail (fallback_col):", df["trail"].nunique())

# 3.3) Pairs & Summary generieren
comps, summary = generate_pairwise_comparisons_from_df(df, num_individuals=4)


print("  Summary der Trail-Längen:")
display(summary)

# 3.4) Resume-Logik für species-spezifisches Parquet
results_fp = output_root / f"{species}_all_results_combined.parquet"
if results_fp.exists():
    done = pd.read_parquet(results_fp, columns=sampled_grid.columns)
    done_keys = set(tuple(r) for r in done.values)
    todo = sampled_grid[
        sampled_grid.apply(lambda row: tuple(row.values) not in done_keys, axis=1)
    ].reset_index(drop=True)
else:
    todo = sampled_grid.copy()

print(f"  → Noch offen: {len(todo)}/{len(sampled_grid)} Runs")

# 3.5) Hyper-Parameter-Loop
for idx, row in todo.iterrows():
    # Spalten heißen jetzt "feature_count" (nicht feature_counts)
    n_features = int(row.feature_count)
    selection = row.selection_method
    reducer = row.reducer
    n_components = int(row.n_components)
    outlier_method = row.outlier_method
    scaler_method = row.scaler_method
    use_sex_flag = bool(row.use_sex)

    desc = (
        f"FC={n_features}|FS={selection}|Red={reducer}/{n_components}"
        f"|Out={outlier_method}|Scl={scaler_method}|Sex={'on' if use_sex_flag else 'off'}"
    )
    print(f"    → Run {idx+1}/{len(todo)}: {desc}")
    t0 = time.time()

    # Outlier- und Scaler-Parameter in Listen verpacken
    outlier_list = [outlier_method] if outlier_method is not None else [None]
    scaler_list = [scaler_method] if scaler_method is not None else [None]

    with tqdm_joblib(tqdm(desc=desc, total=len(comps), leave=False)):
        results = run_all_pairwise_projections_parallel(
            comparisons=comps,
            df=df,
            feature_cols=feature_cols,
            k_features=n_features,
            selection_method=selection,
            reducers=[reducer],
            n_components=[n_components],
            outlier_methods=outlier_list,
            scaler_methods=scaler_list,
            use_sexmodel_prediction=use_sex_flag,
            sexmodel_path=(models_root / f"{species}.joblib") if use_sex_flag else None,
            debug=False,
            n_jobs=-1,
        )
    print(f"      → {len(results)} Rows in {time.time()-t0:.1f}s")

    # 3.6) Results anreichern & speichern
    df_run = pd.DataFrame(results)
    for col in sampled_grid.columns:
        df_run[col] = row[col]
    if results_fp.exists():
        df_full = pd.concat([pd.read_parquet(results_fp), df_run], ignore_index=True)
    else:
        df_full = df_run
    df_full.to_parquet(results_fp, index=False)

print(f"  ✅ Fertig für {species}, Parquet: {results_fp}")

# %% 4) Ergebnisse
all_parquets = [results_fp]
df_all = pd.concat((pd.read_parquet(p) for p in all_parquets), ignore_index=True)
df_all.to_parquet(
    output_root / "eurasian_otter_all_results_combined.parquet", index=False
)
print("\n🎉 Fertig – Ergebnis-Parquet erzeugt:")
print(f"   {output_root/'eurasian_otter_all_results_combined.parquet'}")
display(df_all.head())

c:\Users\Frede\miniconda3\envs\FIT_python_conda_env\Lib\site-packages\tqdm_joblib\__init__.py:4: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


ImportError: cannot import name 'generate_pairwise_comparisons_from_df' from 'FIT_python.pipeline_individual_id.generate_trails_and_trailpairs' (C:\otter_fit\FIT_package\src\FIT_python\pipeline_individual_id\generate_trails_and_trailpairs.py)

In [ ]:
df_all

In [ ]:
# %% Embedding-Once Pipeline Evaluation Block (extended k and reducers)

import time
from pathlib import Path
import pandas as pd
from tqdm.notebook import tqdm
from tqdm_joblib import tqdm_joblib

from FIT_python.pipeline_individual_id.pairwise_individual_id_pipeline import (
    run_embedding_once_pipeline,
)

# ————————————————————————————————————————————————————————————————
# Assumptions:
#   • df           = your full DataFrame of trails
#   • feature_cols = list of feature column names
#   • comps        = output of generate_pairwise_comparisons_from_df(df, ...)
#   • models_root, output_root, species defined as before
# ————————————————————————————————————————————————————————————————

# Define the expanded parameter grid:
k_features_list = [10, 12, 14, 16, 18, 20, 30, 50, 70, 100, 200]
reducers = ["lda", "pca", "umap"]
n_components = 2  # fixed for LDA; adjust if you wish to sweep further

results_fp = output_root / f"{species}_embedding_once_expanded.parquet"

all_runs = []
run_id = 0
total_runs = len(k_features_list) * len(reducers) * 2  # ×2 for use_sex False/True

for kf in k_features_list:
    for red in reducers:
        for use_sex_flag in [False, True]:
            run_id += 1
            desc = (
                f"[{run_id}/{total_runs}] k={kf} | reducer={red}/{n_components} "
                f"| sex={'on' if use_sex_flag else 'off'}"
            )
            print(desc)
            t0 = time.time()

            with tqdm_joblib(tqdm(desc=desc, total=len(comps), leave=False)):
                run_results = run_embedding_once_pipeline(
                    comparisons=comps,
                    df=df,
                    feature_cols=feature_cols,
                    k_features=kf,
                    selection_method="forward",
                    reducer=red,
                    n_components=n_components,
                    outlier_method=None,
                    scaler_method=None,
                    debug=False,
                )

            elapsed = time.time() - t0
            print(f"  → {len(run_results)} comparisons in {elapsed:.1f}s")

            df_run = pd.DataFrame(run_results)
            df_run["k_features"] = kf
            df_run["reducer"] = red
            df_run["n_components"] = n_components
            df_run["use_sex"] = use_sex_flag
            df_run["runtime_s"] = elapsed

            all_runs.append(df_run)

# concatenate and save
df_all = pd.concat(all_runs, ignore_index=True)
df_all.to_parquet(results_fp, index=False)

print(f"\n🎉 Expanded embedding-once results saved to:\n  {results_fp}")
display(df_all.head())

[1/66] k=10 | reducer=lda/2 | sex=off


[1/66] k=10 | reducer=lda/2 | sex=off:   0%|          | 0/300 [00:00<?, ?it/s]

  0%|          | 0/300 [00:00<?, ?it/s]

  → 300 comparisons in 2.3s
[2/66] k=10 | reducer=lda/2 | sex=on


[2/66] k=10 | reducer=lda/2 | sex=on:   0%|          | 0/300 [00:00<?, ?it/s]

...

  → 300 comparisons in 2.3s
[5/66] k=10 | reducer=umap/2 | sex=off


[5/66] k=10 | reducer=umap/2 | sex=off:   0%|          | 0/300 [00:00<?, ?it/s]

  0%|          | 0/300 [00:00<?, ?it/s]

c:\Users\Frede\anaconda3\envs\FIT_python_conda_env\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\Frede\anaconda3\envs\FIT_python_conda_env\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


ValueError: could not convert string to float: 'japet'

In [ ]:
df_all